# 🧾 Notebook 2: Event Sourcing + CQRS

In Notebook 1 we stored **current state** (users, orders) and derived a projection.
**Event sourcing** flips that on its head:

> Don't store *state*. Store the **list of things that happened**.
> Rebuild any view you want by **replaying** the events.

Analogy: a **bank account** doesn't remember only your balance — it keeps
every deposit and withdrawal. The balance is just `sum(events)`. If you ever
need the balance at the end of last March, you replay up to that date. ✨

What you get:

- 🟢 **Free audit log** — every change is recorded by design.
- 🟢 **New views for free** — invent a new projection, replay history, done.
- 🟢 **Time-travel debugging** — replay events up to any point.
- 🟡 Trade-off: more plumbing, events become part of your schema (be careful changing them).


## 🛠️ Setup

```bash
cd 05-microservices/cqrs
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1️⃣ The event log

We'll keep events as tiny dicts. In a real system they'd live in Kafka,
EventStoreDB, or a `Postgres` append-only table.

In [ ]:
from typing import Callable
import time

events: list[dict] = []             # our append-only log
projections: list[Callable] = []    # functions that consume events

def emit(event: dict):
    '''Append one event and notify every projection.'''
    event = {**event, 'ts': time.time()}
    events.append(event)
    for proj in projections:
        proj(event)


## 2️⃣ Write side: commands produce events

A command like "place an order" is **validated**, then turned into one or more
events. The events are the **facts** — the command might be rejected, but once
an event is stored, it is history.

In [ ]:
# Minimal write-side state, used only to validate commands.
known_users: set = set()

def cmd_create_user(uid, name):
    if uid in known_users:
        raise ValueError('user exists')
    known_users.add(uid)
    emit({'type': 'UserCreated', 'uid': uid, 'name': name})

def cmd_place_order(oid, uid, total):
    if uid not in known_users:
        raise ValueError('unknown user')
    if total <= 0:
        raise ValueError('total must be positive')
    emit({'type': 'OrderPlaced', 'oid': oid, 'uid': uid, 'total': total})

def cmd_cancel_order(oid):
    emit({'type': 'OrderCancelled', 'oid': oid})


## 3️⃣ Read side: projections consume events

Each projection is a function that reacts to events and updates its own view.
Different reports = different projections, all fed from the same log.

In [ ]:
#  Rule of thumb: each projection owns its own state. They never
# read another projection's memory -- they only consume events.

# Projection A: per-user summary (dashboard)
user_summary = {}
_summary_orders = {}   # private to p_summary: oid -> (uid, total)

def p_summary(e):
    t = e['type']
    if t == 'UserCreated':
        user_summary[e['uid']] = {'name': e['name'], 'orders': 0, 'spent': 0.0}
    elif t == 'OrderPlaced':
        u = user_summary[e['uid']]
        u['orders'] += 1
        u['spent']  += e['total']
        _summary_orders[e['oid']] = (e['uid'], e['total'])
    elif t == 'OrderCancelled':
        o = _summary_orders.pop(e['oid'], None)
        if o:
            uid, total = o
            u = user_summary[uid]
            u['orders'] -= 1
            u['spent']  -= total

# Projection B: top spenders leaderboard -- fully independent of p_summary.
top_spenders = {}
_top_orders = {}       # private to p_top

def p_top(e):
    if e['type'] == 'OrderPlaced':
        top_spenders[e['uid']] = top_spenders.get(e['uid'], 0) + e['total']
        _top_orders[e['oid']] = (e['uid'], e['total'])
    elif e['type'] == 'OrderCancelled':
        o = _top_orders.pop(e['oid'], None)
        if o:
            uid, total = o
            top_spenders[uid] = top_spenders.get(uid, 0) - total

projections.extend([p_summary, p_top])

# Drive some commands
cmd_create_user(1, 'Ada')
cmd_create_user(2, 'Grace')
cmd_place_order(101, 1, 42.0)
cmd_place_order(102, 2, 99.0)
cmd_place_order(103, 1, 8.0)
cmd_cancel_order(102)

print('summary     :', user_summary)
print('top spenders:', sorted(top_spenders.items(), key=lambda x: -x[1]))
print('event count :', len(events))


## 4️⃣ The superpower: add a new projection, replay history

It's Monday, the product manager asks for *"average order size per user"*.
We didn't plan for it — but the **events are still there**. Define a projection,
replay the log. Zero migrations.

In [ ]:
avg_by_user = {}

def p_avg(e):
    if e['type'] == 'OrderPlaced':
        a = avg_by_user.setdefault(e['uid'], {'n': 0, 'sum': 0.0})
        a['n']  += 1
        a['sum'] += e['total']

# Replay ALL past events into the new projection.
for ev in events:
    p_avg(ev)

# Wire it up for future events too.
projections.append(p_avg)

print({u: round(a['sum']/a['n'], 2) for u, a in avg_by_user.items() if a['n']})


## 5️⃣ Snapshots — keeping replays fast

If you have millions of events, replaying from zero is slow. The standard trick
is **snapshots**: save the projection's state every N events; on startup, load
the latest snapshot and only replay events after it.

In [ ]:
import json, copy

def snapshot(state):
    return json.dumps({'at_event': len(events), 'state': state})

def restore(snap_json):
    snap = json.loads(snap_json)
    return snap['at_event'], snap['state']

# Take a snapshot of the summary projection "right now".
snap = snapshot(copy.deepcopy(user_summary))
print('snapshot bytes:', len(snap))

# --- Time passes, new events arrive ---
cmd_place_order(201, 1, 7.0)
cmd_place_order(202, 2, 3.0)

# --- Fresh service starts up: restore snapshot, replay only NEW events ---
at, restored_state = restore(snap)
# JSON serialises int keys as strings; convert back.
rebuilt = {int(k): v for k, v in restored_state.items()}

# Replay every event after index `at`. In production each projection would
# store and advance its own cursor ("at_event") atomically with its state.
for ev in events[at:]:
    t = ev['type']
    if t == 'OrderPlaced':
        u = rebuilt[ev['uid']]
        u['orders'] += 1
        u['spent']  += ev['total']

print(f'restored at event #{at}, replayed {len(events) - at} newer events')
print('final user 1 :', rebuilt[1])
print('final user 2 :', rebuilt[2])


## ⚠️ Gotchas

- **Event schema is forever.** Adding a field is fine; renaming or removing one
  breaks old events. Version your events (`OrderPlacedV2`) or write *upcasters*.
- **Projections are derived data.** Never edit them by hand — rebuild from events.
- **Order matters.** If projections live on different machines, use the event's
  sequence number (or Kafka partition key) to process events in order per entity.
- **Eventual consistency.** See Notebook 3.


## 📦 The dual-write problem — and the outbox pattern

In the code above, `emit()` appends to the log **and** notifies the projections in one
breath. That works because they live in the same process. In production they don't: the
write model is a database and the projections are fed by a message bus, so the command
handler performs **two writes to two systems with no shared transaction**.

If the second one fails, the read model is wrong *forever* — it isn't "eventually
consistent", it's **permanently** inconsistent, because there is no retry: the event
that would have fixed it was never recorded anywhere.

Let's break it on purpose.

In [ ]:
import random

order_db  = {}     # the write model (a database)
bus_inbox = []     # what the projection actually received (a message broker)

def publish_flaky(event):
    if random.random() < 0.4:                 # the broker is having a bad minute
        raise RuntimeError('broker unreachable')
    bus_inbox.append(event)

def place_order_dual_write(oid, total):
    order_db[oid] = {'total': total}          # write #1: committed to the DB
    publish_flaky({'type': 'OrderPlaced', 'oid': oid, 'total': total})  # write #2

random.seed(3)
lost = 0
for oid in range(20):
    try:
        place_order_dual_write(oid, 10.0)
    except RuntimeError:
        lost += 1

read_model_total  = sum(e['total'] for e in bus_inbox)
write_model_total = sum(o['total'] for o in order_db.values())

print(f'orders committed to the write DB : {len(order_db)}')
print(f'events the read side ever saw    : {len(bus_inbox)}  ({lost} lost forever)')
print(f'revenue per write model          : ${write_model_total:.2f}')
print(f'revenue per read model           : ${read_model_total:.2f}   ← the dashboard lies')
print()
print('Rebuilding the projection would not help: the missing facts are not in any log.')

### ✅ The fix: write the event into an **outbox table, in the same transaction**

1. In **one DB transaction**, write (a) the business change and (b) the event row into an
   `outbox` table *in the same database*. One transaction, so it's all-or-nothing.
2. A separate **relay** reads unpublished outbox rows and pushes them to the bus,
   marking each `published` only after the broker acknowledges.
3. If a publish fails, the row simply stays unpublished and the relay tries again.

The producer's success condition collapses to one question — *did the transaction
commit?* — which a database can actually answer.

In [ ]:
order_db, outbox, bus_inbox = {}, [], []

def place_order_outbox(oid, total):
    # ── pretend this whole block is ONE database transaction ──
    order_db[oid] = {'total': total}
    outbox.append({'id': len(outbox), 'type': 'OrderPlaced',
                   'oid': oid, 'total': total, 'published': False})
    # No publish here. The relay owns that.

def relay_tick():
    for row in outbox:
        if row['published']:
            continue
        try:
            publish_flaky({'type': row['type'], 'oid': row['oid'], 'total': row['total']})
            row['published'] = True      # only after the broker accepted it
        except RuntimeError:
            pass                          # leave it unpublished; retry next tick

random.seed(3)
for oid in range(20):
    place_order_outbox(oid, 10.0)

for tick in range(1, 30):
    relay_tick()
    if all(r['published'] for r in outbox):
        print(f'all 20 events delivered after {tick} relay tick(s)')
        break

print(f'revenue per write model : ${sum(o["total"] for o in order_db.values()):.2f}')
print(f'revenue per read model  : ${sum(e["total"] for e in bus_inbox):.2f}   ← matches ✅')
print()
print(f'events actually delivered to the bus: {len(bus_inbox)} for 20 orders')
print('Note that number can EXCEED 20 in general: the relay may publish a row and')
print('then crash before marking it published, so it republishes on the next tick.')
print('The outbox buys you at-LEAST-once, never exactly-once — which is why every')
print('consumer still needs the event_id dedupe from the EDA lab.')

> In real systems you rarely hand-roll the relay. **Debezium**, **Kafka Connect**, or
> Postgres logical decoding read the database's own change log (`WAL` / `binlog`) and
> publish for you — same guarantee, no polling loop to operate.